# RAG Corpus Daily Refresh
Incremental ingestion of documents into the RAG corpus from multiple sources.

**Outputs**:
- `{catalog}.bronze.rag_corpus` -- unified document store for RAG retrieval

**Sources**:
1. RSS news articles (Yahoo Finance + Google News)
2. SEC EDGAR filing summaries
3. GDELT event summaries
4. Stock context / company profiles

**Schedule**: Daily 9 PM ET

**Parameters**:
| Widget | Default | Description |
|--------|---------|-------------|
| `catalog` | riskbricks | Unity Catalog name |
| `lookback_days` | 3 | Days of content to fetch |
| `max_symbols` | 0 | 0=all from company_universe |

In [0]:
%pip install feedparser beautifulsoup4 --quiet
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "riskbricks", "Catalog Name")
dbutils.widgets.text("lookback_days", "3", "Lookback Days for Content")
dbutils.widgets.text("max_symbols", "0", "Max Symbols (0=all)")
dbutils.widgets.text("edgar_email", "riskbricks@example.com", "SEC EDGAR User-Agent Email")

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import time
import json
import hashlib
import requests
import feedparser
from bs4 import BeautifulSoup
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
)

# -- Read widgets -------------------------------------------------------
catalog = dbutils.widgets.get("catalog").strip()
lookback_days = int(dbutils.widgets.get("lookback_days") or "3")
max_symbols = int(dbutils.widgets.get("max_symbols") or "0")
edgar_email = dbutils.widgets.get("edgar_email").strip()
local_tz = ZoneInfo("America/New_York")
now = datetime.now(local_tz)
cutoff_date = (now - timedelta(days=lookback_days)).strftime("%Y-%m-%d")

# -- Ensure schemas exist -----------------------------------------------
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")

try:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
except Exception:
    pass

# -- Load symbols from company_universe ---------------------------------
universe = spark.sql(f"""
    SELECT DISTINCT symbol, company_name
    FROM {catalog}.gold.company_universe
    ORDER BY symbol
""").collect()

symbols = [row.symbol for row in universe]
symbol_to_name = {row.symbol: row.company_name for row in universe}
if max_symbols and max_symbols > 0:
    symbols = symbols[:max_symbols]

def doc_id(source, symbol, title):
    """Generate deterministic document ID for deduplication."""
    raw = f"{source}|{symbol}|{title}"
    return hashlib.sha256(raw.encode()).hexdigest()[:16]

print(f"Config: catalog={catalog}, lookback_days={lookback_days}")
print(f"Symbols: {len(symbols)}, cutoff: {cutoff_date}")

In [0]:
from email.utils import parsedate_to_datetime

rss_docs = []
for sym in symbols:
    company = symbol_to_name.get(sym, sym)
    feeds = [
        f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={sym}&region=US&lang=en-US",
        f"https://news.google.com/rss/search?q={company}+stock&hl=en-US&gl=US&ceid=US:en",
    ]
    for feed_url in feeds:
        try:
            parsed = feedparser.parse(feed_url)
            for entry in parsed.entries:
                title = entry.get("title", "")
                link = entry.get("link", "")
                summary = entry.get("summary", "")
                # Clean HTML from summary
                if summary:
                    summary = BeautifulSoup(summary, "html.parser").get_text(separator=" ")[:2000]
                pub_date = None
                if entry.get("published"):
                    try:
                        pub_date = parsedate_to_datetime(entry["published"])
                    except Exception:
                        pub_date = now
                else:
                    pub_date = now

                content = f"{title}\n\n{summary}" if summary else title
                rss_docs.append({
                    "doc_id": doc_id("rss", sym, title),
                    "source": "rss_news",
                    "symbol": sym,
                    "title": title[:500],
                    "content": content[:5000],
                    "url": link[:1000],
                    "published_date": pub_date,
                    "ingestion_timestamp": now,
                })
        except Exception:
            pass
    time.sleep(0.1)

print(f"RSS: {len(rss_docs)} articles from {len(symbols)} symbols")

In [0]:
EDGAR_HEADERS = {"User-Agent": f"RiskBricks DataPlatform {edgar_email}"}

sec_docs = []

# Get CIK mapping
try:
    resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=EDGAR_HEADERS, timeout=30)
    resp.raise_for_status()
    cik_data = resp.json()
    cik_map = {}
    for entry in cik_data.values():
        ticker = entry.get("ticker", "").upper()
        cik = entry.get("cik_str")
        if ticker in symbols and cik:
            cik_map[ticker] = str(cik).zfill(10)
except Exception as e:
    print(f"CIK lookup failed: {e}")
    cik_map = {}

for sym in symbols:
    cik = cik_map.get(sym)
    if not cik:
        continue
    try:
        url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        resp = requests.get(url, headers=EDGAR_HEADERS, timeout=30)
        if resp.status_code != 200:
            continue
        data = resp.json()
        filings = data.get("filings", {}).get("recent", {})
        forms = filings.get("form", [])
        dates = filings.get("filingDate", [])
        descs = filings.get("primaryDocDescription", [])
        accessions = filings.get("accessionNumber", [])

        for j in range(min(len(forms), 5)):  # last 5 filings
            form_type = forms[j] if j < len(forms) else ""
            if form_type not in ("10-K", "10-Q", "8-K", "DEF 14A"):
                continue
            filed = dates[j] if j < len(dates) else ""
            if filed < cutoff_date:
                continue
            desc = descs[j] if j < len(descs) else ""
            acc = accessions[j] if j < len(accessions) else ""
            sec_url = f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}/{acc.replace('-', '')}/{acc}-index.htm"

            content = f"{symbol_to_name.get(sym, sym)} ({sym}) filed {form_type} on {filed}. {desc}"
            sec_docs.append({
                "doc_id": doc_id("sec", sym, f"{form_type}_{filed}"),
                "source": "sec_edgar",
                "symbol": sym,
                "title": f"{sym} {form_type} ({filed})",
                "content": content[:5000],
                "url": sec_url,
                "published_date": datetime.strptime(filed, "%Y-%m-%d").replace(tzinfo=local_tz) if filed else now,
                "ingestion_timestamp": now,
            })
    except Exception:
        pass
    time.sleep(0.15)

print(f"SEC: {len(sec_docs)} filing summaries")

In [0]:
# Summarize recent GDELT events already in bronze as RAG documents
gdelt_docs = []
gdelt_table = f"{catalog}.bronze.historical_news_gdelt"

if spark.catalog.tableExists(gdelt_table):
    recent_events = spark.sql(f"""
        SELECT symbol, event_date, actor1_name, actor2_name, avg_tone,
               goldstein_scale, num_articles, source_url
        FROM {gdelt_table}
        WHERE event_date >= '{cutoff_date}'
        ORDER BY event_date DESC, num_articles DESC
    """).collect()

    # Group by symbol+date, create summary docs
    from collections import defaultdict
    grouped = defaultdict(list)
    for row in recent_events:
        grouped[(row.symbol, str(row.event_date))].append(row)

    for (sym, dt), events in grouped.items():
        top_events = events[:10]  # top 10 by articles
        summary_lines = []
        for ev in top_events:
            tone_label = "positive" if ev.avg_tone > 0 else "negative" if ev.avg_tone < 0 else "neutral"
            summary_lines.append(
                f"- {ev.actor1_name or 'Unknown'} / {ev.actor2_name or 'Unknown'}: "
                f"tone={ev.avg_tone:.1f} ({tone_label}), {ev.num_articles} articles"
            )
        content = f"GDELT events for {sym} on {dt}:\n" + "\n".join(summary_lines)
        gdelt_docs.append({
            "doc_id": doc_id("gdelt", sym, f"summary_{dt}"),
            "source": "gdelt_summary",
            "symbol": sym,
            "title": f"{sym} GDELT Summary ({dt})",
            "content": content[:5000],
            "url": "",
            "published_date": datetime.strptime(dt, "%Y-%m-%d").replace(tzinfo=local_tz),
            "ingestion_timestamp": now,
        })

    print(f"GDELT summaries: {len(gdelt_docs)} docs from {len(grouped)} symbol-days")
else:
    print(f"GDELT table not found, skipping")

In [0]:
# -- Combine all sources ------------------------------------------------
all_docs = rss_docs + sec_docs + gdelt_docs
print(f"\nTotal documents: {len(all_docs)} (RSS={len(rss_docs)}, SEC={len(sec_docs)}, GDELT={len(gdelt_docs)})")

if not all_docs:
    dbutils.notebook.exit(json.dumps({"status": "skipped", "message": "No documents collected"}))

rag_schema = StructType([
    StructField("doc_id", StringType(), False),
    StructField("source", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("title", StringType(), True),
    StructField("content", StringType(), True),
    StructField("url", StringType(), True),
    StructField("published_date", TimestampType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
])

df = spark.createDataFrame(all_docs, schema=rag_schema)

rag_table = f"{catalog}.bronze.rag_corpus"
if not spark.catalog.tableExists(rag_table):
    df.write.mode("overwrite").partitionBy("source", "symbol").saveAsTable(rag_table)
else:
    # Deduplicate: only append docs with new doc_ids
    existing_ids = spark.sql(f"SELECT DISTINCT doc_id FROM {rag_table}").select("doc_id")
    new_docs = df.join(existing_ids, on="doc_id", how="left_anti")
    new_count = new_docs.count()
    if new_count > 0:
        new_docs.write.mode("append").saveAsTable(rag_table)
        print(f"Appended {new_count} new documents (deduplicated)")
    else:
        print("No new documents to append (all duplicates)")

final_count = spark.sql(f"SELECT COUNT(*) as n FROM {rag_table}").collect()[0].n
print(f"RAG corpus total: {final_count:,} documents")

In [0]:
result = {
    "status": "success",
    "catalog": catalog,
    "lookback_days": lookback_days,
    "rss_docs": len(rss_docs),
    "sec_docs": len(sec_docs),
    "gdelt_docs": len(gdelt_docs),
    "total_docs": len(all_docs),
    "symbols": len(symbols),
}

print("=" * 60)
print("RAG CORPUS DAILY REFRESH COMPLETE")
print("=" * 60)
print(f"  Catalog:       {catalog}")
print(f"  Lookback:      {lookback_days} days")
print(f"  RSS articles:  {len(rss_docs):,}")
print(f"  SEC filings:   {len(sec_docs):,}")
print(f"  GDELT summaries: {len(gdelt_docs):,}")
print(f"  Total ingested: {len(all_docs):,}")
print(f"  Corpus total:  {final_count:,}")

dbutils.notebook.exit(json.dumps(result))